# Batch / multi-layer auto-discovery

Demonstrates `--batch` (gap #2): point the pipeline at a `RawFolder` and it discovers every
matching `{FileStem}_{NNNNNN}{Ext}` file in a numeric range, skips `dark_*`, and runs one
layer per discovered file. The CLI version is

```
midas-ff-pipeline run --params Parameters.txt --result results/ \
    --layers 1-20 --batch
```

Below we drive the same flow programmatically so you can intervene
between layers (status checks, custom logging, etc.).


In [ ]:
from pathlib import Path
import argparse

from midas_ff_pipeline import Pipeline, PipelineConfig
from midas_ff_pipeline.config import LayerSelection
from midas_ff_pipeline.discovery import (
    discover_layer_files,
    run_batch,
)


## 1. Inputs — edit these

Point at a real `RawFolder` containing `{stem}_NNNNNN.ge3` (or other) files.

In [ ]:
PARAMS = Path('/path/to/Parameters.txt')   # FF parameter file (used for FileStem, Ext, Padding, …)
RESULT_DIR = Path('./batch_results')
LAYER_RANGE = (1, 5)                          # inclusive


## 2. Preview which raw files will be processed

`discover_layer_files` is the same scanner the pipeline uses internally.

In [ ]:
def _read(p: Path, key: str, default: str) -> str:
    for raw in p.read_text().splitlines():
        line = raw.split('#', 1)[0].strip().rstrip(';').rstrip()
        if not line:
            continue
        toks = line.split()
        if toks[0] == key and len(toks) >= 2:
            return toks[1].rstrip(';')
    return default

raw_folder = _read(PARAMS, 'RawFolder', '.')
ext = _read(PARAMS, 'Ext', '.ge3')
padding = int(_read(PARAMS, 'Padding', '6'))
start_fn = int(_read(PARAMS, 'StartFileNrFirstLayer', '1'))

found = discover_layer_files(
    raw_folder, ext, padding,
    start_fn + LAYER_RANGE[0] - 1,
    start_fn + LAYER_RANGE[1] - 1,
)
print(f'discovered {len(found)} files')
for fn, stem in found:
    print(f'  file_nr={fn}  stem={stem}')


## 3. Run all discovered layers

`run_batch` patches `FileStem` per layer and invokes `Pipeline._run_layer`
for each — same code path the CLI's `--batch` flag uses.

In [ ]:
config = PipelineConfig(
    result_dir=str(RESULT_DIR),
    params_file=str(PARAMS),
    n_cpus=8,
    device='cpu',
    dtype='float64',
    layer_selection=LayerSelection(LAYER_RANGE[0], LAYER_RANGE[1]),
)
pipe = Pipeline(config=config)

# run_batch reads the layer range from config.layer_selection and walks discovered files.
# Pass an empty argparse.Namespace; only the layer range from config matters.
args = argparse.Namespace()
run_batch(pipe, args)

for r in pipe.layer_results:
    print(f'layer {r.layer_nr}: {r.n_grains} grains in {r.total_duration_s():.1f}s')


## 4. Inspect resume state

Every layer dir keeps a `midas_state.h5` provenance ledger. The same data is
available via `midas-ff-pipeline status <result_dir>`.

In [ ]:
import json
print(json.dumps(pipe.status(), indent=2, default=str))
